In [1]:
import pandas as pd
import psycopg2
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
import os



In [2]:
load_dotenv("/home/nikita/projects/first-project-yandex-mle/airflow/.env")

DB_DESTINATION_HOST = os.getenv("DB_DESTINATION_HOST")
DB_DESTINATION_PORT = os.getenv("DB_DESTINATION_PORT")
DB_DESTINATION_NAME = os.getenv("DB_DESTINATION_NAME")
DB_DESTINATION_USER = os.getenv("DB_DESTINATION_USER")
DB_DESTINATION_PASSWORD = os.getenv("DB_DESTINATION_PASSWORD")

In [3]:

conn = psycopg2.connect(
    host=DB_DESTINATION_HOST,
    port=DB_DESTINATION_PORT,
    dbname=DB_DESTINATION_NAME,
    user=DB_DESTINATION_USER,
    password=DB_DESTINATION_PASSWORD
)

sql = """
SELECT *
FROM cleaned_data_housing
"""

df = pd.read_sql(sql, conn)

conn.close()

df

/tmp/ipykernel_2815844/90319.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,id,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,1,0,6220,9,9.90,19.900000,1,0,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,1
1,2,1,18012,7,0.00,16.600000,1,0,43.000000,13500000.0,2001,2,55.794849,37.608013,3.00,97,10,1
2,3,2,17821,9,9.00,32.000000,2,0,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,1
3,4,4,9293,3,3.00,14.000000,1,0,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,1
4,5,5,23964,9,0.00,0.000000,2,0,51.009998,8490104.0,2017,4,55.724728,37.743069,2.70,192,17,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97173,97174,141356,9503,8,6.00,42.000000,3,0,64.000000,10800000.0,1971,4,55.740402,37.834579,2.64,428,9,1
97174,97175,141358,3162,5,5.28,28.330000,2,0,41.110001,7400000.0,1960,1,55.727470,37.768677,2.48,80,5,0
97175,97176,141359,6513,7,5.30,20.000000,1,0,31.500000,9700000.0,1966,4,55.704315,37.506584,2.64,72,9,1
97176,97177,141360,23952,15,13.80,33.700001,2,0,65.300003,11750000.0,2017,4,55.699863,37.939564,2.70,480,25,1


In [4]:
df.drop(columns=['id', 'flat_id'], inplace=True) 


In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    df,
    df['price']
)

In [6]:
# Тренировочная выборка
cat_features_tr = X_tr[["is_apartment", "building_type_int", "has_elevator"]]
potential_binary_features_tr = cat_features_tr.nunique() == 2

binary_cat_features_tr = cat_features_tr[potential_binary_features_tr[potential_binary_features_tr].index]
other_cat_features_tr = cat_features_tr[potential_binary_features_tr[~potential_binary_features_tr].index]
num_features_tr = X_tr.select_dtypes(['float'])

cat_features_val = X_val[["is_apartment", "building_type_int", "has_elevator"]]
potential_binary_features_val = cat_features_val.nunique() == 2

binary_cat_features_val = cat_features_val[potential_binary_features_val[potential_binary_features_val].index]
other_cat_features_val = cat_features_val[potential_binary_features_val[~potential_binary_features_val].index]
num_features_val = X_val.select_dtypes(['float'])

In [7]:
cat_features_tr["has_elevator"].value_counts()

has_elevator
1    65077
0     7806
Name: count, dtype: int64